# Transcript-Level Benchmarking for Baleen

This notebook evaluates the pipeline's ability to identify **modified sites** (positions) within transcripts.

## Key Differences from Read-Level
- Aggregates read-level probabilities to site-level calls
- Evaluates V1 → V2 → V3 pipeline stages
- Tests hierarchical shrinkage and HMM smoothing effects
- Measures site-level precision/recall rather than read-level

In [ ]:
import sys
from pathlib import Path

if Path.cwd().name == 'notebooks':
    sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score, f1_score
import pandas as pd
import seaborn as sns

from baleen.eventalign._hierarchical import compute_sequential_modification_probabilities
from baleen.eventalign._pipeline import PositionResult, ContigResult

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Synthetic Data Generation for Transcripts

In [ ]:
def make_block_distance_matrix(n_native, n_ivt, within_native=1.0, within_ivt=1.0, 
                               between=5.0, noise=0.1, seed=42):
    rng = np.random.RandomState(seed)
    n = n_native + n_ivt
    mat = np.zeros((n, n), dtype=np.float64)
    for i in range(n):
        for j in range(i + 1, n):
            if i < n_native and j < n_native:
                d = within_native
            elif i >= n_native and j >= n_native:
                d = within_ivt
            else:
                d = between
            d += rng.normal(0, noise)
            d = max(d, 0.0)
            mat[i, j] = d
            mat[j, i] = d
    return mat


def make_homogeneous_matrix(n_native, n_ivt, base_dist=1.0, noise=0.05, seed=42):
    rng = np.random.RandomState(seed)
    n = n_native + n_ivt
    mat = np.zeros((n, n), dtype=np.float64)
    for i in range(n):
        for j in range(i + 1, n):
            d = base_dist + rng.normal(0, noise)
            d = max(d, 0.0)
            mat[i, j] = d
            mat[j, i] = d
    return mat


def make_transcript_result(
    n_positions: int = 50,
    n_native: int = 20,
    n_ivt: int = 15,
    modified_positions: set = None,
    modification_rate: float = 0.1,
    effect_size: float = 5.0,
    noise: float = 0.15,
    seed: int = 42,
) -> tuple:
    """Create a synthetic transcript with known modification sites.
    
    Returns (ContigResult, set of modified position indices)
    """
    rng = np.random.RandomState(seed)
    
    if modified_positions is None:
        # Randomly select modified positions
        n_modified = int(n_positions * modification_rate)
        modified_positions = set(rng.choice(n_positions, n_modified, replace=False))
    
    positions = {}
    native_names = [f"native_{i}" for i in range(n_native)]
    ivt_names = [f"ivt_{i}" for i in range(n_ivt)]
    
    for idx in range(n_positions):
        if idx in modified_positions:
            dm = make_block_distance_matrix(
                n_native, n_ivt,
                within_native=1.5, within_ivt=1.0,
                between=effect_size + 1.0,
                noise=noise,
                seed=seed + idx,
            )
        else:
            dm = make_homogeneous_matrix(
                n_native, n_ivt,
                base_dist=1.0, noise=0.05,
                seed=seed + idx,
            )
        
        positions[idx] = PositionResult(
            position=idx,
            reference_kmer="NNNNA",
            n_native_reads=n_native,
            n_ivt_reads=n_ivt,
            native_read_names=list(native_names),
            ivt_read_names=list(ivt_names),
            distance_matrix=dm,
        )
    
    cr = ContigResult(
        contig="transcript_1",
        native_depth=float(n_native),
        ivt_depth=float(n_ivt),
        positions=positions,
    )
    
    return cr, modified_positions

## 2. Single Transcript Analysis

Run the full V1 → V2 → V3 pipeline on one transcript.

In [ ]:
# Create test transcript
cr, true_mods = make_transcript_result(
    n_positions=100,
    n_native=25,
    n_ivt=15,
    modification_rate=0.1,
    effect_size=5.0,
    noise=0.15,
    seed=42,
)

print(f"Transcript: {cr.contig}")
print(f"Positions: {len(cr.positions)}")
print(f"True modified positions: {len(true_mods)} ({len(true_mods)/len(cr.positions)*100:.1f}%)")

In [ ]:
# Run full pipeline
result = compute_sequential_modification_probabilities(cr)

print(f"\nPipeline completed")
print(f"Native trajectories: {len(result.native_trajectories)}")
print(f"IVT trajectories: {len(result.ivt_trajectories)}")
print(f"Global μ: {result.global_mu:.3f}")
print(f"Global σ: {result.global_sigma:.3f}")

## 3. Site-Level Performance Metrics

In [ ]:
def extract_site_level_scores(result, true_mods):
    """Extract per-position scores and labels."""
    positions = sorted(result.position_stats.keys())
    
    data = {
        'position': [],
        'y_true': [],
        'mean_z_score': [],
        'mean_p_mod_raw': [],
        'mean_p_mod_knn': [],
        'mean_p_mod_hmm': [],
        'gate_weight': [],
        'coverage_class': [],
    }
    
    for pos in positions:
        ps = result.position_stats[pos]
        data['position'].append(pos)
        data['y_true'].append(1 if pos in true_mods else 0)
        data['mean_z_score'].append(np.mean(ps.native_z_scores))
        data['mean_p_mod_raw'].append(np.mean(ps.native_p_mod_raw))
        data['mean_p_mod_knn'].append(np.nanmean(ps.native_p_mod_knn))
        data['mean_p_mod_hmm'].append(np.nanmean(ps.native_p_mod_hmm))
        data['gate_weight'].append(ps.gate_weight)
        data['coverage_class'].append(ps.coverage_class.value)
    
    return pd.DataFrame(data)

site_df = extract_site_level_scores(result, true_mods)
site_df.head(10)

In [ ]:
# ROC curves for each pipeline stage
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

score_cols = ['mean_z_score', 'mean_p_mod_raw', 'mean_p_mod_knn', 'mean_p_mod_hmm']
labels = ['V1: Z-Score', 'V2: P(mod) Raw', 'V2: P(mod) kNN', 'V3: P(mod) HMM']
colors = ['#9C27B0', '#4CAF50', '#2196F3', '#FF5722']

for ax, col, label, color in zip(axes, score_cols, labels, colors):
    y_true = site_df['y_true'].values
    scores = site_df[col].values
    
    # Handle NaN in HMM column
    valid = ~np.isnan(scores)
    y_true_valid = y_true[valid]
    scores_valid = scores[valid]
    
    fpr, tpr, _ = roc_curve(y_true_valid, scores_valid)
    roc_auc = auc(fpr, tpr)
    pr_auc = average_precision_score(y_true_valid, scores_valid)
    
    ax.plot(fpr, tpr, '-', color=color, lw=2, label=f'ROC (AUC={roc_auc:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'{label}\nPR-AUC={pr_auc:.3f}')
    ax.legend(loc='lower right')
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])

plt.suptitle('Site-Level ROC Curves by Pipeline Stage', y=1.02)
plt.tight_layout()
plt.savefig('transcript_level_pipeline_stages.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. V1 → V2 → V3 Improvement Analysis

In [ ]:
# Compare improvement across stages
stage_metrics = []

for col, label in zip(score_cols, labels):
    y_true = site_df['y_true'].values
    scores = site_df[col].values
    valid = ~np.isnan(scores)
    
    roc = auc(*roc_curve(y_true[valid], scores[valid])[:2])
    pr = average_precision_score(y_true[valid], scores[valid])
    
    # Best F1
    thresholds = np.linspace(scores[valid].min(), scores[valid].max(), 100)
    f1s = [f1_score(y_true[valid], scores[valid] > t) for t in thresholds]
    best_f1 = max(f1s)
    
    stage_metrics.append({
        'Stage': label,
        'ROC-AUC': roc,
        'PR-AUC': pr,
        'Best F1': best_f1,
    })

stage_df = pd.DataFrame(stage_metrics)
stage_df

In [ ]:
# Bar plot comparison
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(stage_df))
width = 0.25

bars1 = ax.bar(x - width, stage_df['ROC-AUC'], width, label='ROC-AUC', color='#2196F3')
bars2 = ax.bar(x, stage_df['PR-AUC'], width, label='PR-AUC', color='#4CAF50')
bars3 = ax.bar(x + width, stage_df['Best F1'], width, label='Best F1', color='#FF9800')

ax.set_ylabel('Score')
ax.set_xlabel('Pipeline Stage')
ax.set_title('Site-Level Performance by Pipeline Stage')
ax.set_xticks(x)
ax.set_xticklabels(['V1\n(Z-Score)', 'V2\n(P Raw)', 'V2\n(P kNN)', 'V3\n(P HMM)'])
ax.legend()
ax.set_ylim([0, 1.1])

# Add value labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('transcript_level_stage_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Effect Size Sweep at Site Level

In [ ]:
effect_sizes = np.linspace(2.0, 10.0, 9)
n_trials = 3

effect_results = {
    'V1_z': [], 'V2_raw': [], 'V2_knn': [], 'V3_hmm': []
}

for effect in effect_sizes:
    trial_aucs = {k: [] for k in effect_results}
    
    for trial in range(n_trials):
        cr, true_mods = make_transcript_result(
            n_positions=50,
            n_native=20, n_ivt=12,
            modification_rate=0.1,
            effect_size=effect,
            noise=0.15,
            seed=trial * 1000 + int(effect * 10),
        )
        result = compute_sequential_modification_probabilities(cr)
        site_df = extract_site_level_scores(result, true_mods)
        y_true = site_df['y_true'].values
        
        for key, col in zip(['V1_z', 'V2_raw', 'V2_knn', 'V3_hmm'], score_cols):
            scores = site_df[col].values
            valid = ~np.isnan(scores)
            if len(np.unique(y_true[valid])) < 2:
                continue
            trial_aucs[key].append(auc(*roc_curve(y_true[valid], scores[valid])[:2]))
    
    for key in effect_results:
        effect_results[key].append(np.mean(trial_aucs[key]) if trial_aucs[key] else np.nan)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

colors = {'V1_z': '#9C27B0', 'V2_raw': '#4CAF50', 'V2_knn': '#2196F3', 'V3_hmm': '#FF5722'}
labels = {'V1_z': 'V1: Z-Score', 'V2_raw': 'V2: P(mod) Raw', 
          'V2_knn': 'V2: P(mod) kNN', 'V3_hmm': 'V3: P(mod) HMM'}

for key in effect_results:
    ax.plot(effect_sizes, effect_results[key], 'o-', 
            color=colors[key], label=labels[key], lw=2, markersize=8)

ax.set_xlabel('Effect Size (Native-IVT Separation)')
ax.set_ylabel('Site-Level ROC-AUC')
ax.set_title('Site-Level Performance vs Effect Size')
ax.legend()
ax.set_ylim([0.4, 1.05])

plt.tight_layout()
plt.savefig('transcript_level_effect_size.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Modification Rate Impact

In [ ]:
mod_rates = [0.02, 0.05, 0.1, 0.15, 0.2, 0.3]
n_trials = 3

rate_results = {'V3_hmm': [], 'V2_knn': []}

for rate in mod_rates:
    trial_aucs = {k: [] for k in rate_results}
    
    for trial in range(n_trials):
        cr, true_mods = make_transcript_result(
            n_positions=100,
            n_native=20, n_ivt=12,
            modification_rate=rate,
            effect_size=5.0,
            noise=0.15,
            seed=trial * 100 + int(rate * 1000),
        )
        result = compute_sequential_modification_probabilities(cr)
        site_df = extract_site_level_scores(result, true_mods)
        y_true = site_df['y_true'].values
        
        for key, col in [('V3_hmm', 'mean_p_mod_hmm'), ('V2_knn', 'mean_p_mod_knn')]:
            scores = site_df[col].values
            valid = ~np.isnan(scores)
            if len(np.unique(y_true[valid])) < 2:
                continue
            trial_aucs[key].append(auc(*roc_curve(y_true[valid], scores[valid])[:2]))
    
    for key in rate_results:
        rate_results[key].append(np.mean(trial_aucs[key]) if trial_aucs[key] else np.nan)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ROC-AUC
axes[0].plot(mod_rates, rate_results['V2_knn'], 'o-', color='#2196F3', 
             label='V2: kNN', lw=2, markersize=8)
axes[0].plot(mod_rates, rate_results['V3_hmm'], 's-', color='#FF5722',
             label='V3: HMM', lw=2, markersize=8)
axes[0].set_xlabel('Modification Rate')
axes[0].set_ylabel('Site-Level ROC-AUC')
axes[0].set_title('ROC-AUC vs Modification Rate')
axes[0].legend()
axes[0].set_ylim([0.4, 1.05])

# PR-AUC comparison (conceptual - shows class imbalance effect)
axes[1].bar(['2%', '5%', '10%', '15%', '20%', '30%'], 
           [rate_results['V3_hmm'][i] - rate_results['V2_knn'][i] 
            for i in range(len(mod_rates))],
           color='#4CAF50', alpha=0.7)
axes[1].set_xlabel('Modification Rate')
axes[1].set_ylabel('HMM - kNN (ROC-AUC Difference)')
axes[1].set_title('HMM Improvement over kNN')
axes[1].axhline(y=0, color='k', ls='--', lw=1)

plt.suptitle('Impact of Modification Rate on Site-Level Performance', y=1.02)
plt.tight_layout()
plt.savefig('transcript_level_mod_rate.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. HMM Smoothing Visualization

In [ ]:
# Show HMM smoothing effect along the transcript
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

# Top: probability tracks
ax = axes[0]
positions = site_df['position'].values

ax.fill_between(positions, 0, site_df['y_true'].values, alpha=0.2, color='red', label='True Modified')
ax.plot(positions, site_df['mean_p_mod_knn'], 'o-', color='#2196F3', 
        markersize=4, lw=1, alpha=0.7, label='V2: kNN')
ax.plot(positions, site_df['mean_p_mod_hmm'], 's-', color='#FF5722',
        markersize=4, lw=1, alpha=0.7, label='V3: HMM')

ax.set_ylabel('Mean P(modified)')
ax.set_title('Probability Tracks Along Transcript')
ax.legend(loc='upper right')
ax.set_ylim([-0.05, 1.05])

# Bottom: z-scores
ax = axes[1]
ax.bar(positions, site_df['mean_z_score'], color=np.where(site_df['y_true'], 'red', 'gray'),
       alpha=0.7, width=0.8)
ax.axhline(y=0, color='k', lw=0.5)
ax.axhline(y=2, color='red', ls='--', lw=1, alpha=0.5, label='z=2 threshold')

ax.set_xlabel('Position')
ax.set_ylabel('Mean Z-Score')
ax.set_title('V1 Z-Scores Along Transcript')
ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig('transcript_level_tracks.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Coverage Class Impact

In [ ]:
# Analyze performance by IVT coverage class
coverage_stats = site_df.groupby('coverage_class').agg({
    'y_true': ['sum', 'count'],
    'mean_p_mod_hmm': 'mean',
    'gate_weight': 'mean',
}).round(3)

coverage_stats.columns = ['n_modified', 'n_total', 'mean_p_mod_hmm', 'mean_gate_weight']
coverage_stats['modification_rate'] = (coverage_stats['n_modified'] / coverage_stats['n_total']).round(3)
coverage_stats

In [ ]:
# Box plots by coverage class
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Group by coverage class and true label
for ax, (col, title) in zip(axes, [('mean_p_mod_knn', 'V2: kNN'), ('mean_p_mod_hmm', 'V3: HMM')]):
    data_mod = []
    data_unmod = []
    labels = []
    
    for cc in ['high', 'medium', 'low']:
        subset = site_df[site_df['coverage_class'] == cc]
        data_mod.append(subset[subset['y_true'] == 1][col].dropna().values)
        data_unmod.append(subset[subset['y_true'] == 0][col].dropna().values)
        labels.extend([f'{cc}\nmod', f'{cc}\nunmod'])
    
    positions = []
    data = []
    for i, (d_mod, d_unmod) in enumerate(zip(data_mod, data_unmod)):
        positions.extend([i * 2, i * 2 + 0.5])
        data.extend([d_mod, d_unmod])
    
    bp = ax.boxplot(data, positions=positions, widths=0.35, patch_artist=True)
    
    colors_mod = ['#FF5722', '#FF5722', '#FF5722']
    colors_unmod = ['#2196F3', '#2196F3', '#2196F3']
    for i, patch in enumerate(bp['boxes']):
        if i % 2 == 0:
            patch.set_facecolor(colors_mod[i // 2])
        else:
            patch.set_facecolor(colors_unmod[i // 2])
        patch.set_alpha(0.6)
    
    ax.set_xticks([0.25, 2.25, 4.25])
    ax.set_xticklabels(['HIGH\n(≥20 IVT)', 'MEDIUM\n(5-19 IVT)', 'LOW\n(1-4 IVT)'])
    ax.set_ylabel('P(modified)')
    ax.set_title(title)
    ax.set_ylim([-0.05, 1.05])

plt.suptitle('Score Distributions by Coverage Class', y=1.02)
plt.tight_layout()
plt.savefig('transcript_level_coverage_class.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Summary Statistics

In [ ]:
# Final summary across multiple transcripts
n_transcripts = 10
summary_data = []

for t in range(n_transcripts):
    cr, true_mods = make_transcript_result(
        n_positions=50,
        n_native=20, n_ivt=12,
        modification_rate=0.1,
        effect_size=5.0,
        noise=0.15,
        seed=t * 777 + 42,
    )
    result = compute_sequential_modification_probabilities(cr)
    site_df = extract_site_level_scores(result, true_mods)
    y_true = site_df['y_true'].values
    
    for key, col, label in [('V1', 'mean_z_score', 'Z-Score'),
                            ('V2_knn', 'mean_p_mod_knn', 'kNN'),
                            ('V3_hmm', 'mean_p_mod_hmm', 'HMM')]:
        scores = site_df[col].values
        valid = ~np.isnan(scores)
        if len(np.unique(y_true[valid])) < 2:
            continue
        
        roc = auc(*roc_curve(y_true[valid], scores[valid])[:2])
        pr = average_precision_score(y_true[valid], scores[valid])
        
        summary_data.append({
            'Transcript': t,
            'Stage': label,
            'ROC-AUC': roc,
            'PR-AUC': pr,
        })

summary_df = pd.DataFrame(summary_data)
summary_df.groupby('Stage').agg({'ROC-AUC': ['mean', 'std'], 'PR-AUC': ['mean', 'std']}).round(3)

In [ ]:
# Display summary
print("\n" + "="*60)
print("TRANSCRIPT-LEVEL BENCHMARK SUMMARY")
print("="*60)
print(f"\nConditions:")
print(f"  - Positions per transcript: 50")
print(f"  - Native reads per position: 20")
print(f"  - IVT reads per position: 12")
print(f"  - Modification rate: 10%")
print(f"  - Effect size: 5.0")
print(f"  - Noise: 0.15")
print(f"  - Number of transcripts: {n_transcripts}")
print()
print(summary_df.groupby('Stage').agg({'ROC-AUC': ['mean', 'std'], 'PR-AUC': ['mean', 'std']}).round(3).to_string())